# SpeakEz — real Parakeet transcription on a Colab GPU

This notebook joins the GPU to the demo stack. It only makes **outbound** connections:
an SSH tunnel to Redis + Postgres on the VPS, and HTTPS to the API for the audio.
Nothing connects *into* Colab.

**Before running:** add these secrets in the 🔑 panel (Colab → Secrets), so nothing is pasted
into a shared or screen-recorded cell:

| Secret | What it is |
| --- | --- |
| `SPEAKEZ_REPO_URL` | git clone URL of this repo |
| `SPEAKEZ_SSH_KEY` | the **private** half of the forwarding-only key (`deploy/README.md` §Colab) |
| `SPEAKEZ_VPS_HOST` | VPS IP or hostname |
| `SPEAKEZ_DB_PASSWORD` | the `POSTGRES_PASSWORD` from `.env` on the VPS |
| `SPEAKEZ_API_BASE_URL` | public API URL, e.g. `https://api.<dashed-ip>.sslip.io` |
| `SPEAKEZ_JWT_SECRET` | same as the API — it signs the audio URL the worker downloads |
| `SPEAKEZ_EMAIL_PEPPER` | same as the API |

Use a **GPU runtime** (Runtime → Change runtime type → T4 GPU). Run the cells top to bottom.
A dropped runtime has to redo cells 3–5 (install + ~2.5 GB of weights): budget ~15 minutes, not a refresh.

In [ ]:
#@title 1. Get the code
import os
from google.colab import userdata

REPO = userdata.get('SPEAKEZ_REPO_URL')
if not os.path.isdir('speakez'):
    !git clone {REPO} speakez
%cd speakez
!git pull --ff-only

In [ ]:
#@title 2. Open the tunnel to the VPS (Redis + Postgres)
import os, subprocess
from google.colab import userdata

VPS = userdata.get('SPEAKEZ_VPS_HOST')
os.makedirs('/root/.ssh', exist_ok=True)
os.chmod('/root/.ssh', 0o700)

key = userdata.get('SPEAKEZ_SSH_KEY')
with open('/root/.ssh/id_ed25519', 'w') as handle:
    handle.write(key if key.endswith('\n') else key + '\n')
os.chmod('/root/.ssh/id_ed25519', 0o600)

# Pin the host key so the tunnel can't be MITM'd.
subprocess.run(f'ssh-keyscan -H {VPS} >> /root/.ssh/known_hosts', shell=True, check=True)

# -f backgrounds it, -N runs no command; ExitOnForwardFailure fails loudly if a port is taken.
# ServerAliveInterval keeps an idle tunnel from being dropped while you wait for a judge.
subprocess.run(
    f'ssh -f -N -o ExitOnForwardFailure=yes -o ServerAliveInterval=30 '
    f'-i /root/.ssh/id_ed25519 -L 6379:127.0.0.1:6379 -L 5432:127.0.0.1:5432 root@{VPS}',
    shell=True, check=True,
)
print('tunnel up — worker will use localhost:6379 and localhost:5432')

In [ ]:
#@title 3. Install NeMo WITHOUT touching Colab's torch
# Colab ships a torch matched to its CUDA driver. Installing worker/requirements-gpu.txt
# here would let pip resolve a different build and break torch.cuda.is_available().
# Install the pieces explicitly instead, and DO NOT `pip install torch`.
!apt-get -qq install -y ffmpeg
!pip -q install "nemo_toolkit[asr]" arq httpx
!pip -q install ./api

# First run takes 5–10 min and prints dependency-resolver warnings; that is normal.
# If pip reports it upgraded something already imported, restart the runtime and re-run
# cells 1–2, then continue at cell 4.

In [ ]:
#@title 4. Point the worker at the tunnel
import os
from google.colab import userdata

os.environ.update(
    REDIS_URL='redis://localhost:6379/0',
    DATABASE_URL=(
        'postgresql+asyncpg://speakez:'
        + userdata.get('SPEAKEZ_DB_PASSWORD')
        + '@localhost:5432/speakez'
    ),
    API_BASE_URL=userdata.get('SPEAKEZ_API_BASE_URL'),
    JWT_SECRET=userdata.get('SPEAKEZ_JWT_SECRET'),
    EMAIL_PEPPER=userdata.get('SPEAKEZ_EMAIL_PEPPER'),
    ASR_ENGINE='parakeet',
    HF_HOME='/content/models',
    DEMO_MODE='true',
)
for name in ('REDIS_URL', 'DATABASE_URL', 'API_BASE_URL', 'JWT_SECRET', 'EMAIL_PEPPER'):
    assert os.environ.get(name), name
print('env ready')

In [ ]:
#@title 5. Pre-warm the model (downloads ~2.5 GB — do this before judging)
import os, sys
sys.path.insert(0, os.getcwd())

import torch
print('cuda available:', torch.cuda.is_available())  # must be True

from worker import asr
asr.load_model()
print('Parakeet loaded and cached in', os.environ['HF_HOME'])

In [ ]:
#@title 6. Run the worker (foreground — stop this cell to disconnect)
!python -u deploy/colab_worker.py

## Verify — the only test that counts

Record a note **in the app** saying something specific and unusual ("the seventeenth of November,
raining"), watch `docker compose logs -f api` show `POST /notes` then `POST /notes/{id}/submit`,
and confirm in order:

- [ ] `GET /notes/{id}/draft` returns **your actual sentence**, not the demo string
- [ ] `words` carry real start/end timings (not empty, not uniform)
- [ ] the story screen highlights word-by-word in time with the audio
- [ ] publishing makes the pin appear on the map within ~60 s (the cron worker is alive)
- [ ] **safety**: `DEMO_TRANSCRIPT` is ignored under `parakeet`, so *say* something that trips the
      lexicon and confirm it routes to `/care` and never reaches the map

The last one matters most: the safety gate now runs against real ASR output, a different code
path from the fixed demo string.

**If Colab dies during judging**, put the always-on worker back on the real queue from the VPS:

```bash
docker compose -f docker-compose.prod.yml --profile cpu-worker run -d --rm \
  -e ASR_ENGINE=demo worker arq worker.main.WorkerSettings
```